In [ ]:
# 01 - Setup project and dependencies
from pathlib import Path
import os, sys, subprocess, zipfile, shutil

PROJECT_DIR = Path('/kaggle/working/Spine2Space')
if not (PROJECT_DIR / 'src').exists():
    PROJECT_DIR.mkdir(parents=True, exist_ok=True)
    input_root = Path('/kaggle/input')
    zip_matches = list(input_root.glob('**/spine2space_kaggle_source.zip'))
    source_roots = [p.parent for p in input_root.glob('**/pyproject.toml') if (p.parent / 'src').exists()]
    if zip_matches:
        with zipfile.ZipFile(zip_matches[0]) as archive:
            archive.extractall(PROJECT_DIR)
        print('extracted:', zip_matches[0])
    elif source_roots:
        src_root = source_roots[0]
        for item in src_root.iterdir():
            destination = PROJECT_DIR / item.name
            if item.is_dir():
                shutil.copytree(item, destination, dirs_exist_ok=True)
            else:
                shutil.copy2(item, destination)
        print('copied source root:', src_root)
    else:
        found = sorted(str(p) for p in input_root.glob('*'))
        raise FileNotFoundError('Could not find spine2space_kaggle_source.zip or extracted project under /kaggle/input. Found: ' + str(found))
os.chdir(PROJECT_DIR)
sys.path.insert(0, str(PROJECT_DIR))

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.[model,medical,data,plots,visualization]'], check=True)
subprocess.run([sys.executable, '-m', 'compileall', '-q', 'src'], check=True)
print('ready:', PROJECT_DIR)


In [ ]:
# 02 - Download a CTSpine1K micro-subset and build one manifest
MAX_PAIRS = 50  # micro-dataset target; lower to 20 if Kaggle time/storage is tight

!python -m src.data.download_ctspine --output-dir data/raw/ctspine1k --max-pairs {MAX_PAIRS}
!python -m src.data.manifest --config configs/real_split_prep.yaml --output data/processed/real_split/manifest.jsonl
!python -m src.drr.generator --config configs/real_split_prep.yaml --manifest data/processed/real_split/manifest.jsonl --limit {MAX_PAIRS}


In [ ]:
# 03 - Create patient-level train/validation manifests
!python -m src.data.split_manifest \
  --manifest data/processed/real_split/manifest.jsonl \
  --train-output data/processed/real_split/manifest_train.jsonl \
  --val-output data/processed/real_split/manifest_val.jsonl \
  --val-fraction 0.2 \
  --seed 23

!wc -l data/processed/real_split/manifest_train.jsonl data/processed/real_split/manifest_val.jsonl


In [ ]:
# 04 - Train on train patients and validate on unseen patients
!python -m src.train_real_split --config configs/kaggle_real_split.yaml

import json
metrics = json.load(open('runs/real_split_train/metrics.json'))
print('best:', json.dumps(metrics['best'], indent=2))
print('final:', json.dumps(metrics['final'], indent=2))


In [ ]:
# 05 - Evaluate the best checkpoint on the first held-out validation sample
!python -m src.view_volume \
  --checkpoint runs/real_split_train/best.pt \
  --manifest data/processed/real_split/manifest_val.jsonl \
  --sample-index 0 \
  --output-dir runs/real_split_eval

!python -m src.render3d --npz runs/real_split_eval/volumes/prediction_target.npz --output-dir reports/real_split_assets
!python -m src.mesh3d --npz runs/real_split_eval/volumes/prediction_target.npz --output-dir reports/real_split_assets --sigma 0.55


In [ ]:
# 06 - Package outputs for download
!mkdir -p kaggle_outputs
!cp -r runs/real_split_train runs/real_split_eval reports/real_split_assets kaggle_outputs/
!tar -czf spine2space_real_split_outputs.tar.gz kaggle_outputs

!find kaggle_outputs -maxdepth 3 -type f | sort | sed -n '1,120p'
print('Download: /kaggle/working/Spine2Space/spine2space_real_split_outputs.tar.gz')
